# Piloto — transcrição, diarização e medição

Segunda metade da esteira do passo 4 do `docs/roadmap.md`. Recebe áudio já coletado,
transcreve, diariza e produz as medições que hoje são suposições no cálculo da meta
de volume.

## Por que este notebook não coleta

A tentativa de coletar aqui, em 27/08/2026, resultou em **0 de 51 vídeos**. O YouTube
recusa downloads originados de datacenter, respondendo *"Sign in to confirm you're not
a bot"*. Os endereços do Colab são de datacenter; conexão residencial não sofre o
bloqueio. O mesmo plano de coleta que falha aqui executa normalmente numa máquina
doméstica.

A esteira divide-se, portanto, entre os dois ambientes, cada metade onde funciona:

| Etapa | Onde | Motivo |
|---|---|---|
| Coleta | máquina local | conexão residencial não é bloqueada |
| Transcrição e diarização | Colab | exigem GPU |

**Antes de abrir este notebook**, execute na máquina local:

```
python selecionar_videos.py --piloto --max-canais 2 --saida plano_piloto.json
python coletar_local.py plano_piloto.json
```

e envie a pasta `dataset_raw` resultante para o Google Drive.

## O que este notebook mede

1. **Rendimento por camada** — que fração da duração é fala, descontados vinheta,
   música e silêncio. O planejamento supõe 35% para vox-pop, 60% para rádio e TV e
   70% para vlog. Se o rendimento real do vox-pop for metade do suposto, a meta de
   50 h dobra.
2. **Locutores por arquivo** — na camada de vox-pop é o que separa o morador
   entrevistado do repórter, e disso depende a camada inteira.
3. **Indicador aproximado de dificuldade de transcrição** — confiança média por
   palavra, por estado. **Não é WER**, e o notebook não o apresenta como tal.

**Antes de rodar:** ative a GPU em *Ambiente de execução -> Alterar o tipo de ambiente*.

## 1. Verificação do ambiente

In [ ]:
import shutil

def checar(nome, condicao, detalhe=""):
    print(f"{'OK   ' if condicao else 'FALHA'}  {nome}  {detalhe}")
    return condicao

ok = True
try:
    import torch
    ok &= checar("GPU", torch.cuda.is_available(),
                 torch.cuda.get_device_name(0) if torch.cuda.is_available()
                 else "sem GPU - ative em Ambiente de execucao")
except ImportError:
    print("torch ainda nao instalado; rode a proxima celula e volte aqui")
    ok = False

ok &= checar("ffmpeg", shutil.which("ffmpeg") is not None)

try:
    import faster_whisper, pyannote.audio, numpy, numba
    ok &= checar("faster-whisper e pyannote", True,
                 f"numpy {numpy.__version__}, numba {numba.__version__}")
except Exception as e:
    print(f"FALHA  faster-whisper/pyannote  {type(e).__name__}: {e}")
    print("       Se for conflito de numpy com numba, rode:  !pip install -q 'numpy<2.3'")
    print("       e reinicie a sessao em Ambiente de execucao -> Reiniciar sessao.")
    ok = False

print("\nAmbiente pronto." if ok else "\nCorrija os itens acima antes de prosseguir.")

## 2. Instalação

O limite de `numpy` é deliberado. A instalação sem restrição traz `numpy` mais novo do
que o `numba` aceita, e o `numba` entra por baixo do `pyannote`, via `librosa`. O
conflito só se manifestaria na diarização, isto é, **depois** de a transcrição inteira
já ter rodado.

In [ ]:
!pip install -q "numpy<2.3" "faster-whisper>=1.0.0" "pyannote.audio>=3.1.0" jiwer
!apt-get -qq install -y ffmpeg > /dev/null
print("instalado. Reinicie a sessao em Ambiente de execucao -> Reiniciar sessao,")
print("e execute novamente a celula de verificacao.")

## 3. Repositório, credencial e Drive

O `HF_TOKEN` precisa ter aceitado os termos de
`pyannote/speaker-diarization-community-1`. Use o painel de segredos do Colab, ícone de
chave na barra lateral, com o nome `HF_TOKEN`. Nunca cole o token numa célula: o
notebook é versionado e a célula preserva o conteúdo.

In [ ]:
!git clone -q https://github.com/Aryazinha/vies-nordeste-bertimbau.git

import os
from google.colab import userdata, drive
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
print("HF_TOKEN carregado e Drive montado")

## 4. Localização do material coletado

Ajuste `PASTA_DRIVE` para onde você enviou a pasta `dataset_raw`. A célula confere
que áudio e metadados estão presentes e coerentes entre si — arquivo sem metadado é
áudio sem procedência regional, e procedência é a variável do estudo.

In [ ]:
import json
from pathlib import Path

PASTA_DRIVE = Path("/content/drive/MyDrive/dataset_raw")   # ajuste se necessario

AUDIO = PASTA_DRIVE / "audio"
meta_path = PASTA_DRIVE / "metadados.json"

assert AUDIO.is_dir(), f"pasta de audio nao encontrada: {AUDIO}"
assert meta_path.exists(), f"metadados.json nao encontrado em {PASTA_DRIVE}"

metadados = json.loads(meta_path.read_text(encoding="utf-8"))
wavs = {p.name for p in AUDIO.glob("*.wav")}

sem_audio = [m["id"] for m in metadados if m["arquivo"] not in wavs]
sem_meta = wavs - {m["arquivo"] for m in metadados}

print(f"{len(metadados)} registros de metadados")
print(f"{len(wavs)} arquivos de audio")
if sem_audio:
    print(f"AVISO  {len(sem_audio)} metadados sem audio: {sem_audio[:5]}")
if sem_meta:
    print(f"AVISO  {len(sem_meta)} audios sem metadado, serao ignorados: {list(sem_meta)[:5]}")

from collections import Counter
print("\npor estado:", dict(Counter(m["estado_alvo"] for m in metadados)))
print("por camada:", dict(Counter(m["tipo_fonte"] for m in metadados)))

## 5. Transcrição e diarização

Etapa dominante em tempo. Na primeira execução, o modelo de transcrição é baixado —
cerca de 3 GB — e a célula permanece vários minutos sem imprimir nada, o que é
esperado.

In [ ]:
import sys, json
sys.path.insert(0, "/content/vies-nordeste-bertimbau/pipeline_coleta_piloto")

from transcribe import transcrever_audio
from diarize import diarizar_audio, atribuir_locutor

SAIDA = Path("/content/registros_finais")
SAIDA.mkdir(exist_ok=True)

registros = []
for i, meta in enumerate(metadados, 1):
    caminho = AUDIO / meta["arquivo"]
    if not caminho.exists():
        continue
    print(f"[{i}/{len(metadados)}] {meta['id']}  {meta['estado_alvo']}/{meta['tipo_fonte']}")
    try:
        transcricao = transcrever_audio(caminho)
        turnos = diarizar_audio(caminho)
    except Exception as e:
        print(f"    FALHA: {type(e).__name__}: {e}")
        continue

    for segmento in transcricao["segmentos"]:
        for w in segmento["words"]:
            w["speaker"] = atribuir_locutor(w["start"], w["end"], turnos)

    registro = dict(meta)
    registro["transcricao"] = transcricao
    registro["diarizacao"] = turnos
    (SAIDA / f"{meta['id']}.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
    registros.append(registro)

print(f"\n{len(registros)}/{len(metadados)} registros completos")

## 6. Medições

### 6.1 Rendimento por camada

In [ ]:
from collections import defaultdict

rend = defaultdict(list)
for r in registros:
    segs = r["transcricao"]["segmentos"]
    if not segs:
        continue
    dur = segs[-1]["end"]
    fala = sum(s["end"] - s["start"] for s in segs)
    if dur:
        rend[r["tipo_fonte"]].append(fala / dur)

SUPOSTO = {"entrevista_vox_pop": 0.35,
           "podcast_radio_tv_regional": 0.60,
           "vlog_amador": 0.70}

print(f"{'camada':34s} {'n':>3s} {'medido':>8s} {'suposto':>8s}")
for camada, vals in rend.items():
    print(f"{camada:34s} {len(vals):3d} {sum(vals)/len(vals):8.1%} {SUPOSTO.get(camada, 0):8.1%}")

print("\nDesconta silencio e musica, nao a fala de locutor de outra variedade.")
print("O desconto do reporter aparece na secao 6.2.")

### 6.2 Locutores por arquivo

Arquivo de vox-pop com um único locutor indica falha da diarização ou vídeo sem
entrevista — nos dois casos, material que não serve.

In [ ]:
for r in registros:
    tempos = defaultdict(float)
    for t in r["diarizacao"]:
        tempos[t["speaker"]] += t["end"] - t["start"]
    total = sum(tempos.values()) or 1
    dist = ", ".join(f"{s}={d/total:.0%}"
                     for s, d in sorted(tempos.items(), key=lambda x: -x[1])[:4])
    print(f"{r['estado_alvo']}/{r['tipo_fonte'][:12]:12s} {r['id']}  "
          f"{len(tempos)} locutor(es)  [{dist}]")

por_camada = defaultdict(list)
for r in registros:
    por_camada[r["tipo_fonte"]].append(len({t["speaker"] for t in r["diarizacao"]}))
print()
for camada, ns in por_camada.items():
    print(f"{camada:34s} media de {sum(ns)/len(ns):.1f} locutores por arquivo")

### 6.3 Indicador aproximado de dificuldade de transcrição

Confiança média por palavra, por estado. **Não é WER.** Mede a certeza do modelo, não
o acerto. Presta-se a uma pergunta apenas: existe diferença sistemática entre
variedades que justifique o custo da transcrição manual de referência? Diferença
observada aqui é resultado a investigar, jamais a reportar como WER.

In [ ]:
conf = defaultdict(list)
for r in registros:
    for seg in r["transcricao"]["segmentos"]:
        for w in seg["words"]:
            conf[r["estado_alvo"]].append(w["probability"])

print(f"{'estado':8s} {'palavras':>9s} {'confianca media':>17s}")
for uf in ["PB", "PE", "CE", "BA", "SP", "RJ"]:
    if conf[uf]:
        print(f"{uf:8s} {len(conf[uf]):9d} {sum(conf[uf])/len(conf[uf]):17.3f}")

ne = [p for uf in ("PB", "PE", "CE", "BA") for p in conf[uf]]
se = [p for uf in ("SP", "RJ") for p in conf[uf]]
if ne and se:
    mne, mse = sum(ne)/len(ne), sum(se)/len(se)
    print(f"\nNordeste {mne:.3f}  |  Sudeste {mse:.3f}  |  diferenca {mne-mse:+.3f}")

### 6.4 Amostra para transcrição manual

Exporta trechos totalizando 20 minutos por estado, com `referencia_manual` em branco.
É o insumo do WER propriamente dito.

In [ ]:
import random
random.seed(20260827)

por_estado = defaultdict(list)
for r in registros:
    for seg in r["transcricao"]["segmentos"]:
        if seg["end"] - seg["start"] >= 5:
            por_estado[r["estado_alvo"]].append({
                "id": r["id"], "estado": r["estado_alvo"], "camada": r["tipo_fonte"],
                "inicio_s": round(seg["start"], 2), "fim_s": round(seg["end"], 2),
                "hipotese_asr": seg["text"].strip(), "referencia_manual": "",
            })

selecao = []
for uf, itens in por_estado.items():
    random.shuffle(itens)
    acumulado, escolhidos = 0.0, []
    for i in itens:
        if acumulado >= 20 * 60:
            break
        escolhidos.append(i)
        acumulado += i["fim_s"] - i["inicio_s"]
    selecao.extend(escolhidos)
    print(f"{uf}: {len(escolhidos)} trechos, {acumulado/60:.1f} min")

with open("/content/amostra_wer.json", "w", encoding="utf-8") as f:
    json.dump(selecao, f, ensure_ascii=False, indent=2)
print(f"\n{len(selecao)} trechos em amostra_wer.json")

## 7. Exportação

Os registros são gravados também no Drive, de modo que o resultado sobrevive ao
encerramento da sessão.

In [ ]:
import shutil

destino_drive = PASTA_DRIVE.parent / "piloto_resultados"
shutil.copytree(SAIDA, destino_drive, dirs_exist_ok=True)
shutil.copy("/content/amostra_wer.json", destino_drive)
print(f"gravado em {destino_drive}")

shutil.make_archive("/content/piloto_resultados", "zip", SAIDA)
from google.colab import files
files.download("/content/piloto_resultados.zip")
files.download("/content/amostra_wer.json")